In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from PIL import Image
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, BatchNormalization, LeakyReLU

In [ ]:
img_size = (256, 256)

directory = '/Users/user/Desktop/ВКР_ИТМО/АНОМАЛИИ_ДАТАСЕТ/metal_nut_тест/Все'
directory_clear = '/Users/user/Desktop/ВКР_ИТМО/АНОМАЛИИ_ДАТАСЕТ/metal_nut_тест/Чистые'


glob = {'image_paths': []}
glob_clear = {'image_paths': []}

def extract_image_info(directory):
    image_paths = [
        os.path.join(directory, f) 
        for f in os.listdir(directory) 
        if os.path.isfile(os.path.join(directory, f)) and f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]
    return image_paths

glob['image_paths'] = extract_image_info(directory)
glob_clear['image_paths'] = extract_image_info(directory_clear)

In [ ]:
class ConvAutoencoder:
    def __init__(self, input_shape):
        self.input_shape = input_shape
        self.model = self._build_model()

    def _conv_block(self, x, filters):
        x = Conv2D(filters, (3, 3), padding='same')(x)
        x = BatchNormalization()(x)
        x = LeakyReLU(negative_slope=0.1)(x)
        x = Conv2D(filters, (3, 3), padding='same')(x)
        x = BatchNormalization()(x)
        x = LeakyReLU(negative_slope=0.1)(x)
        return x

    def _build_model(self):
        input_img = Input(shape=self.input_shape)

        # Encoder
        x = self._conv_block(input_img, 64)
        x = MaxPooling2D((2, 2), padding='same')(x)

        x = self._conv_block(x, 128)
        x = MaxPooling2D((2, 2), padding='same')(x)

        x = self._conv_block(x, 256)
        encoded = MaxPooling2D((2, 2), padding='same')(x)

        # Decoder 
        x = UpSampling2D((2, 2))(encoded)
        x = self._conv_block(x, 128)      
        x = UpSampling2D((2, 2))(x)
        x = self._conv_block(x, 64)

        x = UpSampling2D((2, 2))(x)
        x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
        output = Conv2D(self.input_shape[-1], (3, 3), activation='sigmoid', padding='same')(x)

        model = Model(input_img, output)
        model.compile(optimizer=Adam(learning_rate=1e-3), loss=self._ssim_loss)
        return model

    def _ssim_loss(self, y_true, y_pred):
        return 1 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

    def train(self, x_train, epochs=20, batch_size=8):
        return self.model.fit(
            x_train, x_train,
            epochs=epochs,
            batch_size=batch_size,
            shuffle=True,
            verbose=1
        )

    def predict(self, X_test, batch_size=32):
        return self.model.predict(X_test, batch_size=batch_size, verbose=0)

In [ ]:
# Загрузка и предобработка изображений
feature_train_transformed = []
for img_path in glob['image_paths']:
    img = load_img(img_path, target_size=img_size)
    img_array = img_to_array(img)
    feature_train_transformed.append(img_array)

feature_train_transformed = np.array(feature_train_transformed).astype('float32') / 255.0

feature_train_transformed_clear = []
for img_path in glob_clear['image_paths']:
    img = load_img(img_path, target_size=img_size)
    img_array = img_to_array(img)
    feature_train_transformed_clear.append(img_array)

feature_train_transformed_clear = np.array(feature_train_transformed_clear).astype('float32') / 255.0


In [ ]:
# Загрузка и предобработка изображений
feature_train_transformed = []
for img_path in glob['image_paths']:
    img = load_img(img_path, target_size=img_size)
    img_array = img_to_array(img)
    feature_train_transformed.append(img_array)


feature_train_transformed_clear = []
for img_path in glob_clear['image_paths']:
    img = load_img(img_path, target_size=img_size)
    img_array = img_to_array(img)
    feature_train_transformed_clear.append(img_array)

feature_train_transformed_clear = np.array(feature_train_transformed_clear).astype('float32') / 255.0
feature_train_transformed = np.array(feature_train_transformed).astype('float32') / 255.0



In [ ]:
# Обучение
AE = ConvAutoencoder(input_shape=(256, 256, 3))
history = AE.train(x_train=feature_train_transformed, epochs=15, batch_size=8)

In [ ]:
# предсказание реконструкции для всех изображений
reconstructions = AE.predict(feature_train_transformed, batch_size=32)
errors = np.mean(np.square(feature_train_transformed - reconstructions), axis=(1, 2, 3))

# индексы изображений с наибольшими ошибками
top_anomaly_indices = np.argsort(errors)[-20000:][::-1]

In [ ]:
# выведение топа самых вероятно аномальных изображений
for i in range(len(top_anomaly_indices)):
    file_path=glob['image_paths'][top_anomaly_indices[i]]
    if 'Аномалия_' in glob['image_paths'][top_anomaly_indices[i]].split('/')[-1]:

        print(str('№')+'_'+str(i+1),' ' ,glob['image_paths'][top_anomaly_indices[i]].split('/')[-1])

        img = Image.open(file_path).convert("RGB")

        plt.imshow(img)
        plt.axis('off') 
        plt.show()